# API с IGDB

## Контекст:
IGDB - это база данных игр, которая доступна по API.
**Почти тоже самое, что и IMDB, только для игр...**

Через нее монжо получать структурированные поля:
|заполнить|поля|позднее|
|-|-|-|
|жанры|2|1|

Доступ к IGDB возможен только по токену, который мы успешно сгенерировали с помощью Twitch Developer. Подробнее о том, как это сделать можно узнать в [документации IGDB](https://api-docs.igdb.com/#getting-started)

В следующих этапах проекта, мы объединим данные с API со скреппингом (либо используем иначе).

## Начнем с настройки API
Чтобы наши прелестные ключики не улетели в свободное использование на github (так как репозиторий публичен), подтянем их из окружения, до этого закинув их в терминал

In [2]:
import os

In [ ]:
os.environ['twitch_client_secret'] = "67sixseven"
os.environ['twitch_client_id'] = "67sixseven"

#пж замените на данные в тгшке, но не пуште с секретками

In [5]:
cid = os.environ['twitch_client_id']
secret = os.environ['twitch_client_secret']

In [6]:
# а теперь получим bearer token для доступа к API IGDB
import requests
import json
resp = requests.post('https://id.twitch.tv/oauth2/token', data={
    'client_id': cid,
    'client_secret': secret,
    'grant_type': 'client_credentials'
})

data = resp.json()
# data

In [8]:
# data['access_token']

{'access_token': 'сиксевен676767',
 'expires_in': 5283769,
 'token_type': 'bearer'}

In [ ]:
os.environ['access_token'] = 'сиксевен676767'
# os.getenv('access_token')

В С Ë. Мы получили токен для работы с API. Дальше будем работать напрямую с докой и искать данные

### Всю последующую работу я делаю опираясь на Seminar 8 - API

Далее сделаем request. В основном из методов в документации только post. Обращаемся к `https://api.igdb.com/v4`. 

Еще я добавил логгирование чтобы отслеживать важные моменты и отлавливать места с проблемаии

In [11]:
import logging 
logging.basicConfig(level=logging.INFO)

In [12]:
token = os.getenv('access_token')
cid

'po0syvw9g7dh0et8fdhse6y12l4gmu'

In [13]:
# этот url нам понадобится для запросов к API IGDB
url = 'https://api.igdb.com/v4/games'

# в headers идентифицируем себя и указываем, что хотим получить json
headers = {
    'Client-ID': cid,
    'Authorization': f"Bearer {os.getenv('access_token')}",
    'Accept': 'application/json'
}

# а теперь делаем запрос к API IGDB, чтобы получить 10 игр с их рейтингами
body = 'fields name, rating; limit 10;'

logging.info("Отправляем запрос к API IGDB...")
#добавил еще таймаут, чтобы не зависало на запросе
games10 = requests.post(url, headers=headers, data=body, timeout=10)
logging.info(f'Ответ от IGDB {games10.status_code}')

INFO:root:Отправляем запрос к API IGDB...
INFO:root:Ответ от IGDB 200


УРААААА!!! Ответ 200, у нас есть data. Давайте посмотрим как там внутри

In [15]:
games10.json()

[{'id': 350392, 'name': 'Rival Species'},
 {'id': 20243, 'name': 'Wipeout 2', 'rating': 75.0},
 {'id': 207571, 'name': 'A Very Corporate Escape Room'},
 {'id': 339266, 'name': 'Power Guy World'},
 {'id': 63844, 'name': 'Ace wo Nerae!', 'rating': 52.90462943179914},
 {'id': 371149, 'name': 'The Deal'},
 {'id': 330684, 'name': 'Nightmare Kart: The Old Karts'},
 {'id': 371776, 'name': 'Born for the Light: Wavelength Radiant Collapse'},
 {'id': 129477, 'name': 'Chomp Chomp'},
 {'id': 179243,
  'name': 'Satella-Q: Mou Sugu Haru desu ne - Chotto Sate-Q Shimasen ka?'}]

Обратим внимание, что рейтинг есть не везде. Покопавшись чуть глубже в доке узнал, что поле с рейтингом является опциональном. Вообще, покопавшись в метриках можно заметить, что там целый набор метрик: rating, rating_count, aggregated_rating, aggregated_rating_count, total_rating и total_rating_count. 

В таком случае имеет место быть заселектить разные поля, а самое главное поставить фильтры... Я не до конца понимаю по какому принципу сортируется сейчас, поэтому лучше добавить фильтры и сортировку...